In [1]:
!pip install sentence-transformers chromadb groq pandas -q
print("Installation complete!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [2]:
import pandas as pd
import chromadb

from sentence_transformers import SentenceTransformer
from groq import Groq
import os

print("All libraries imported successfully!")

All libraries imported successfully!


In [3]:
GROQ_API_KEY="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"
os.environ["GROQ_API_KEY"]=GROQ_API_KEY
groq_client=Groq(api_key=GROQ_API_KEY)
print("Groq API client intialized.")
print("Note: If you see an authentication error later,double-check your API key")

Groq API client intialized.
Note: If you see an authentication error later,double-check your API key


In [4]:
df=pd.read_csv("college_notes.csv")
print("Shape of datasets:",df.shape)
print("\n Coulmns name:",df.columns.tolist())
print("\n First 3 rows:")
print(df.head(3))

Shape of datasets: (15, 4)

 Coulmns name: ['note_id', 'subject', 'topic', 'content']

 First 3 rows:
  note_id           subject          topic  \
0    N001  Data Engineering  ETL Pipelines   
1    N002  Data Engineering  SQL Databases   
2    N003  Data Engineering  Data Cleaning   

                                             content  
0  ETL stands for Extract Transform Load. It is t...  
1  A database is an organized collection of data ...  
2  Data cleaning involves fixing or removing inco...  


In [5]:
print("Subjects in datasets:")
print(df['subject'].value_counts())

print("\n Sample topic:")
print(df[['note_id','subject','topic']].to_string(index=False))
print("\n Length of content (number of characters) for each note:")
df['content_length']=df['content'].apply(len)


print(df[['topic','content_length']].to_string(index=False))


Subjects in datasets:
subject
Data Engineering      5
Machine Learning      5
Generative AI         3
Python Programming    2
Name: count, dtype: int64

 Sample topic:
note_id            subject                          topic
   N001   Data Engineering                  ETL Pipelines
   N002   Data Engineering                  SQL Databases
   N003   Data Engineering                  Data Cleaning
   N004   Data Engineering       APIs and Data Collection
   N005   Data Engineering           Big Data and PySpark
   N006   Machine Learning            Supervised Learning
   N007   Machine Learning               Model Evaluation
   N008   Machine Learning            Feature Engineering
   N009   Machine Learning                 Decision Trees
   N010   Machine Learning                  Random Forest
   N011      Generative AI          Large Language Models
   N012      Generative AI             Prompt Engineering
   N013      Generative AI Retrieval Augmented Generation
   N014 Python Progr

In [6]:
documents = df['content'].tolist()
ids = [f"note_{row['note_id']}" for row in df.to_dict('records')]
metadatas = [
    {"subject" : row['subject'], "topic": row['topic']}
    for row in df.to_dict('records')
]

print(f"Total Chunks Prepared: {len(documents)}")
print(f"First document ID : {ids[0]}")
print(f"First metadata: {metadatas[0]}")
print(f"first 100 chars of doc: {documents[0][:100]}...")


Total Chunks Prepared: 15
First document ID : note_N001
First metadata: {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
first 100 chars of doc: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc...


In [7]:
print("Loading embedding model...")
print("(This may take 30-60 seconds on first run - model is being downloaded)")
print("(Subsequent runs will be faster as the model is cached)")

embedding_model=SentenceTransformer('all-MiniLM-L6-v2')
print("\nEmbedding model loaded successfully.")
test_embedding=embedding_model.encode("This is a test sentence.")
print(f"Test embedding shape:{test_embedding.shape}")
print(f"First 5 values of test embedding:{test_embedding[:5]}")

Loading embedding model...
(This may take 30-60 seconds on first run - model is being downloaded)
(Subsequent runs will be faster as the model is cached)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Embedding model loaded successfully.
Test embedding shape:(384,)
First 5 values of test embedding:[0.08429647 0.05795366 0.00449333 0.1058211  0.00708344]


In [8]:
chroma_client=chromadb.Client()
collection=chroma_client.get_or_create_collection(name="college_notes_rag")
print("ChromaDb client created")
print(f"Collection name:collecge_notes_rag")
print(f"Documents in collection so far {collection.count()}")

ChromaDb client created
Collection name:collecge_notes_rag
Documents in collection so far 0


In [9]:
print("Generating embedding for all 15 notes..")
print("This may take 15-30 seconds")

embeddings=embedding_model.encode(documents,show_progress_bar=True)
print(f"\n Embedding matrix shape:{embeddings.shape}")

embeddings_list=embeddings.tolist()

collection.add(
    documents=documents,
    ids=ids,
    metadatas=metadatas,
    embeddings=embeddings_list
)

print(f"\nDocuments successfully added to chromaDB")
print(f"Total documents in collection: {collection.count()}")


Generating embedding for all 15 notes..
This may take 15-30 seconds


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


 Embedding matrix shape:(15, 384)

Documents successfully added to chromaDB
Total documents in collection: 15


In [10]:
def retrieve_relevant_chunks(question,top_k=3):
  """
  Given a user question,retrieve the most relevant document chunks from chromadb.

  Parameters:
    questions(str):the user's question as sa text string
    top_k(int):how many top results to return(default:3)

  Returns:
   A dictionary containing retrieved documents,distance and metadata
   """

  question_embedding=embedding_model.encode(question).tolist()
  results=collection.query(
  query_embeddings=[question_embedding],
  n_results=top_k
)
  return results

print("Retrieval function defined successfully")
print("Function:retrieve_relevant_chunks(question,top_k=3)")


Retrieval function defined successfully
Function:retrieve_relevant_chunks(question,top_k=3)


In [14]:
test_question ="what is Etl and how doies it work in data engineering?"
print(f"Test question:{test_question}")
print("=" *60)

results=retrieve_relevant_chunks(test_question,top_k=3)
print("\n Top 3 retrieved chunks:")
print("*" * 60)

for i , (doc,dist,meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
    )):
  print(f"\n Results {i+1}:")
  print(f"Subject :{meta['subject']}")
  print(f"Topic:{meta['topic']}")
  print(f"Distance:{dist:.4f}")
  print(f"Content:{doc[:200]}...")


Test question:what is Etl and how doies it work in data engineering?

 Top 3 retrieved chunks:
************************************************************

 Results 1:
Subject :Data Engineering
Topic:ETL Pipelines
Distance:0.2578
Content:ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehou...

 Results 2:
Subject :Data Engineering
Topic:APIs and Data Collection
Distance:1.0291
Content:An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data from external services like weather data stock price...

 Results 3:
Subject :Python Programming
Topic:Data Visualization
Distance:1.3008
Content:Data visualization is the process of representing data as charts graphs and visual formats. Python libraries like Matplotlib and Seaborn are used to create bar charts li

In [18]:
test_question ="what is Etl and how does it work in data engineering?"
print(f"Test question:{test_question}")
print("=" *60)

results=retrieve_relevant_chunks(test_question,top_k=3)
print("\n Top 3 retrieved chunks:")
print("*" * 60)

for i , (doc,dist,meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
    )):
  print(f"\n Results {i+1}:")
  print(f"Subject :{meta['subject']}")
  print(f"Topic:{meta['topic']}")
  print(f"Distance:{dist:.4f}")
  print(f"Content:{doc[:200]}...")

Test question:what is Etl and how does it work in data engineering?

 Top 3 retrieved chunks:
************************************************************

 Results 1:
Subject :Data Engineering
Topic:ETL Pipelines
Distance:0.2269
Content:ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehou...

 Results 2:
Subject :Data Engineering
Topic:APIs and Data Collection
Distance:1.0690
Content:An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data from external services like weather data stock price...

 Results 3:
Subject :Python Programming
Topic:Data Visualization
Distance:1.3375
Content:Data visualization is the process of representing data as charts graphs and visual formats. Python libraries like Matplotlib and Seaborn are used to create bar charts lin

In [19]:
def build_context_from_results(results):
  """
  Fromat ChromaDB retrieval results into a readable context string.

  parameters:
    results : The output from collection.query() - a dictionary

  Returns:
    context_str(str) : A formatted string of allretrieved dictionary
  """

  context_parts = []

  for i, (doc, meta) in enumerate(zip(
      results['documents'][0],
      results['metadatas'][0]
  )):
    chunk_text = f"[Source{i+1}: {meta['subject']}](doc)"

In [20]:
def build_context_from_results(results):
  """
    Format ChromaDB retrieval results into a readable context string.

    Parameters:
        results: The output from collection.query() - a dictionary

    Returns:
        context_str (str): A formatted string of all retrieved document chunks
    """
  context_parts = []

  for i, (doc, meta) in enumerate(zip(
      results['documents'][0],
      results['metadatas'][0]
  )):
    context_text = f"[Source {i+1}: {meta['subject']} - {meta['topic']}\n{doc}]"
    context_parts.append(context_text)

    return context_parts

In [21]:
def generate_rag_answer(question, context):
    """
    Generate an answer using RAG (Retrieval-Augmented Generation).

    Parameters:
        question (str): User's question
        context (str): Retrieved context from ChromaDB

    Returns:
        answer (str): Generated answer
    """

    system_prompt = """
    You are a helpful academic assistant for engineering students.
    Answer the question using only the provided context.
    If the answer is not in the context, say so.
    """

    user_prompt = f"""
    Context:

[Source 1: ETL - Data Cleaning]
ETL stands for Extract Transform Load and is used in data engineering.

Question:
What is ETL?

Answer:
    """

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.1,
        max_tokens=500
    )

    answer = response.choices[0].message.content

    return answer


print("RAG generation function defined successfully")

RAG generation function defined successfully
